# Aula 6 - RAG

## Versão 6.0 do projeto: um assistente que consulta documentos antes de responder

Nesta aula, vamos dar ao assistente acesso a uma base de documentos. Em vez de depender só do que o modelo aprendeu no treinamento, a aplicação recupera os trechos mais relevantes e os envia junto com a pergunta.

Ao final, você deve conseguir:

- dividir documentos em chunks;
- indexar os chunks e buscar os mais parecidos com uma pergunta;
- montar um prompt aumentado com o contexto recuperado;
- fazer o modelo responder citando as fontes e admitindo quando não sabe;
- expor a busca como uma ferramenta do agente;
- explicar a evolução do projeto da V1.0 até a V6.0.

## 1. Configuração do ambiente

Instale as dependências pelo terminal, seguindo o `README.md` desta pasta. Desta vez, além de `openai` e `python-dotenv`, usaremos `scikit-learn` para a busca. Copie `env.example` para `.env` e preencha `GROQ_API_KEY`. Nunca coloque a chave diretamente neste arquivo nem faça commit do `.env`.

In [ ]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

## 2. Conheça a base de documentos

Toda aplicação RAG começa com uma coleção de documentos. Aqui, a base são notas resumidas sobre cada aula desta oficina. Em um projeto real, esses textos viriam de arquivos, de um banco de dados ou de um site.

In [ ]:
DOCUMENTS = [
    {
        "fonte": "aula1_llm_apis",
        "texto": (
            "Uma chamada a uma LLM API envia uma lista de mensagens com os papéis system, user e assistant "
            "e recebe uma resposta gerada pelo modelo. O modelo não tem memória automática entre chamadas: "
            "a aplicação precisa reenviar o histórico da conversa a cada requisição. O uso é medido em tokens, "
            "que contam tanto o texto de entrada quanto o texto de saída. Parâmetros como temperature controlam "
            "o quanto a resposta varia entre execuções."
        ),
    },
    {
        "fonte": "aula2_context_engineering",
        "texto": (
            "Context Engineering é o trabalho de selecionar e organizar o que entra na janela de contexto antes "
            "da chamada ao modelo. Contexto demais aumenta custo e pode distrair o modelo, contexto de menos "
            "faz o modelo inventar. Boas práticas incluem separar instruções de dados, colocar só o material "
            "relevante e deixar claro para o modelo de onde cada informação veio."
        ),
    },
    {
        "fonte": "aula3_tool_calling",
        "texto": (
            "Em tool calling, a aplicação descreve funções para o modelo com nome, descrição e JSON Schema dos "
            "parâmetros. O modelo não executa nada: ele devolve um tool_call pedindo a execução. A aplicação "
            "valida o nome, converte os argumentos, executa a função permitida e devolve o resultado em uma "
            "mensagem com papel tool, usando o tool_call_id. Uma segunda chamada produz a resposta final."
        ),
    },
    {
        "fonte": "aula4_mcp_sdks",
        "texto": (
            "O Model Context Protocol (MCP) padroniza como aplicações descobrem e usam ferramentas, recursos e "
            "prompts. Um MCP server expõe capacidades e um MCP client, dentro da aplicação, se conecta a ele. "
            "Assim as ferramentas deixam de ser definidas dentro de cada aplicação e passam a ser reutilizadas "
            "por qualquer cliente compatível. SDKs abstraem detalhes de HTTP, autenticação e formatos de resposta."
        ),
    },
    {
        "fonte": "aula5_agents",
        "texto": (
            "Um agente executa um loop de decisão: o modelo decide o próximo passo, a aplicação executa a "
            "ferramenta pedida, o resultado volta ao histórico e o ciclo se repete até que o modelo produza "
            "a resposta final. É essencial definir um limite máximo de iterações para evitar loops infinitos "
            "e registrar cada passo para facilitar a depuração."
        ),
    },
    {
        "fonte": "aula6_rag",
        "texto": (
            "RAG, ou Retrieval-Augmented Generation, combina recuperação e geração. Os documentos são divididos "
            "em chunks e indexados. Na hora da pergunta, a aplicação busca os chunks mais similares e os "
            "inclui no prompt. O RAG reduz alucinações e permite responder sobre dados que o modelo nunca viu, "
            "mas não as elimina: se a recuperação trouxer trechos errados, a resposta também sairá errada."
        ),
    },
]

print(f"{len(DOCUMENTS)} documentos carregados")

## 3. Divida os documentos em chunks

Buscar em documentos inteiros é impreciso e desperdiça contexto. Por isso, cada documento é dividido em trechos menores. A sobreposição entre chunks evita cortar uma ideia exatamente no meio.

In [ ]:
# TODO: implemente `dividir_em_chunks(texto, tamanho=40, sobreposicao=10)`.
# 1. separe o texto em palavras com texto.split();
# 2. avance pela lista de palavras em passos de (tamanho - sobreposicao);
# 3. junte cada janela de palavras em uma string;
# 4. retorne a lista de chunks.

In [ ]:
# TODO: monte a lista `CHUNKS`.
# Cada item deve ser um dicionário com:
# {"id": "aula3_tool_calling#0", "fonte": "aula3_tool_calling", "texto": "..."}
# Percorra DOCUMENTS, divida cada texto e numere os chunks de cada fonte.

# print(f"{len(CHUNKS)} chunks criados")
# print(CHUNKS[0])

## 4. Indexe os chunks

Para comparar uma pergunta com os chunks, transformamos cada texto em um vetor numérico. Nesta aula usamos TF-IDF, que pontua palavras pela frequência nos textos e funciona sem depender de outra API. Em produção, é comum usar embeddings, que capturam o significado além das palavras exatas. A estrutura do restante do código continua a mesma.

In [ ]:
# TODO: crie o índice.
# 1. crie `vectorizer = TfidfVectorizer(strip_accents="unicode", lowercase=True)`;
# 2. ajuste com vectorizer.fit_transform([chunk["texto"] for chunk in CHUNKS]);
# 3. guarde o resultado em `chunk_vectors`.

## 5. Implemente a busca

A busca transforma a pergunta no mesmo espaço vetorial e ordena os chunks por similaridade. Retornamos também a pontuação, porque ela ajuda a perceber quando nada na base responde à pergunta.

In [ ]:
# TODO: implemente `buscar_documentos(pergunta, k=3)`.
# 1. transforme a pergunta com vectorizer.transform([pergunta]);
# 2. calcule cosine_similarity(query_vector, chunk_vectors)[0];
# 3. ordene os índices do maior para o menor score;
# 4. retorne os k melhores como lista de dicionários com id, fonte, score e texto.

In [ ]:
# Teste a busca antes de ligar o modelo:
# for item in buscar_documentos("Como o modelo pede a execução de uma função?"):
#     print(round(item["score"], 3), item["id"])
#     print(item["texto"], "\n")

## 6. Pergunte sem RAG

Antes de aumentar o prompt, veja o comportamento do modelo sem acesso à base. Escolha uma pergunta cuja resposta esteja nos documentos, como os detalhes da Aula 5.

In [ ]:
query = "[DIGITE AQUI SUA PERGUNTA]"

# TODO: chame client.chat.completions.create com apenas a pergunta do usuário
# (sem tools e sem contexto) e guarde em `answer_without_rag`.

# print(answer_without_rag.choices[0].message.content)

## 7. Monte o prompt aumentado

Agora a aplicação recupera os trechos e os coloca no prompt. Duas instruções fazem diferença: responder apenas com base no contexto e admitir quando a informação não está lá.

In [ ]:
# TODO: implemente `montar_mensagens(pergunta, trechos)`.
# 1. formate os trechos como texto, indicando a fonte de cada um:
#    "[aula3_tool_calling#0] texto do trecho"
# 2. crie a mensagem system pedindo para:
#    - responder somente com base no contexto fornecido;
#    - citar as fontes entre colchetes;
#    - dizer que não encontrou a informação quando o contexto não bastar;
# 3. coloque o contexto e a pergunta na mensagem user, separados por marcadores claros;
# 4. retorne a lista de mensagens.

## 8. Faça a chamada com o contexto

Recupere os trechos, monte as mensagens e chame o modelo. Imprima também os trechos usados, para poder conferir se a resposta está de acordo com o que foi recuperado.

In [ ]:
# TODO: execute o fluxo completo com a mesma `query` da seção 6.
# 1. trechos = buscar_documentos(query, k=3)
# 2. messages = montar_mensagens(query, trechos)
# 3. answer_with_rag = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
# 4. imprima os ids dos trechos e a resposta.

## 9. Transforme a busca em uma ferramenta

Na seção anterior, a busca acontece sempre. Combinando com o que vimos nas Aulas 3 e 5, podemos deixar o modelo decidir quando consultar a base: `buscar_documentos` vira uma ferramenta. Perguntas conceituais gerais dispensam a busca, e perguntas sobre o material da oficina a acionam.

In [ ]:
# TODO: crie o schema da ferramenta `buscar_documentos` seguindo o formato da Aula 3.
# Parâmetros: `pergunta` (string, obrigatório) e `k` (integer, opcional).
# Depois registre a função em AVAILABLE_TOOLS.
#
# tools = [ { "type": "function", "function": { ... } } ]
# AVAILABLE_TOOLS = {"buscar_documentos": buscar_documentos}

In [ ]:
# Exemplo de loop com limite de iterações (Aula 5) usando a busca como ferramenta:
# messages = [
#     {"role": "system", "content": (
#         "Você é um assistente da oficina de IA. Para perguntas sobre o conteúdo das aulas, "
#         "use buscar_documentos e cite as fontes. Se a busca não trouxer a resposta, diga que não encontrou."
#     )},
#     {"role": "user", "content": query},
# ]
#
# MAX_STEPS = 5
# for step in range(MAX_STEPS):
#     response = client.chat.completions.create(
#         model=MODEL, messages=messages, tools=tools, tool_choice="auto", temperature=0,
#     )
#     message = response.choices[0].message
#     messages.append(message)
#
#     if not message.tool_calls:
#         print(message.content)
#         break
#
#     for call in message.tool_calls:
#         name = call.function.name
#         if name not in AVAILABLE_TOOLS:
#             raise ValueError(f"Ferramenta não permitida: {name}")
#         arguments = json.loads(call.function.arguments)
#         result = AVAILABLE_TOOLS[name](**arguments)
#         messages.append({
#             "role": "tool",
#             "tool_call_id": call.id,
#             "content": json.dumps(result, ensure_ascii=False),
#         })
# else:
#     print("Limite de iterações atingido.")

## 10. Compare os três caminhos

Faça três perguntas e observe como cada abordagem se comporta:

1. uma pergunta cuja resposta está na base;
2. uma pergunta conceitual geral, que não depende da base;
3. uma pergunta sobre algo que **não está** na base (por exemplo, "Qual é a data da prova final?").

Na terceira, o comportamento esperado é o assistente dizer que não encontrou a informação, em vez de inventar.

In [ ]:
# TODO: preencha a tabela abaixo com o que observou.
#
# | Pergunta | Sem RAG | Com RAG (sempre busca) | Com RAG como tool |
# |----------|---------|------------------------|-------------------|
# | 1        |         |                        |                   |
# | 2        |         |                        |                   |
# | 3        |         |                        |                   |

In [ ]:
# Se a resposta ficou ruim, investigue a recuperação antes de culpar o modelo:
# for item in buscar_documentos(query, k=3):
#     print(round(item["score"], 3), item["id"])
#
# Perguntas para se fazer:
# - o chunk certo apareceu entre os k resultados?
# - o score do melhor chunk é muito baixo quando a base não tem a resposta?
# - mudar o tamanho do chunk ou o valor de k melhora o resultado?

## 11. Checkpoint da V6

A V6 está pronta quando:

- os documentos são divididos em chunks com sobreposição;
- o índice é criado e a busca retorna os chunks mais similares com score;
- o prompt aumentado inclui o contexto e pede citação de fontes;
- o assistente admite quando a base não contém a resposta;
- `buscar_documentos` está exposta como ferramenta com validação de nome;
- o loop respeita um limite de iterações;
- a chave continua no `.env`;
- as alterações foram salvas em um commit separado.